# U-Sleep RBD/PD:156–159, наша схема019

Официальный Pretrained_Model.ckpt; наш preprocessing019 по непрерывным сегментам, core128/halo64, среднее двух EEG/EOG-пар.
Без high-pass, Hann и дополнительного full-context прохода. Затем смеси с сохранённым113.
Input тот же: исходные данные + sleep-edf-usleep-rbd-inputs.zip. T4×2, Internet On для research launcher.
RUN_GROUP="all", SMOKE_ONLY=False: полный val/test, четыре CSV и накопительный ZIP. Старые результаты этой серии не возобновлять.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/revlze/sleep-edf-classification.git"
CODE_REF = "feat/usleep-rbd-pd"
PROJECT_ROOT = Path("/tmp/sleep-edf-usleep-rbd-code-v2")
OUTPUT_ROOT = Path("/kaggle/working/usleep-rbd-blocks-results-v2")
INPUT_ROOT = None  # Автопоиск нового sleep-edf-usleep-rbd-inputs.zip.
RESUME_ROOT = None  # Необязательно: результаты ЭТОЙ серии и версии кода.
RUN_GROUP = "all"  # all / standalone
SMOKE_ONLY = False  # Сразу полный val/test; True только для структурного smoke.
SMOKE_EPOCHS = 4
DEVICE = "cuda:0"
THREADS = max(1, os.cpu_count() or 2)


def checkout_code():
    if PROJECT_ROOT.exists():
        current_ref = subprocess.check_output(
            ["git", "config", "sleepedf.bootstrapRef"], cwd=PROJECT_ROOT, text=True
        ).strip()
        if current_ref != CODE_REF:
            raise RuntimeError(
                "Для другого CODE_REF задайте новый PROJECT_ROOT в /tmp."
            )
        print(
            "Используется существующий checkout; для обновления ветки задайте новый PROJECT_ROOT."
        )
        return
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory(prefix="sleep-git-") as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            'print("x-access-token" if "Username" in sys.argv[1] else os.environ["SLEEP_GITHUB_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        env = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "SLEEP_GITHUB_TOKEN": token,
        }
        subprocess.run(
            ["git", "init", "--quiet", "--initial-branch=bootstrap", str(PROJECT_ROOT)],
            check=True,
        )
        subprocess.run(
            ["git", "remote", "add", "origin", REPO_URL], cwd=PROJECT_ROOT, check=True
        )
        subprocess.run(
            [
                "git",
                "-c",
                "credential.helper=",
                "fetch",
                "--quiet",
                "--depth",
                "1",
                "origin",
                CODE_REF,
            ],
            cwd=PROJECT_ROOT,
            env=env,
            check=True,
        )
        subprocess.run(
            ["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"],
            cwd=PROJECT_ROOT,
            check=True,
        )
        subprocess.run(
            ["git", "config", "sleepedf.bootstrapRef", CODE_REF],
            cwd=PROJECT_ROOT,
            check=True,
        )
    del token


checkout_code()
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
print("Код:", commit)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook import reset_commands, run_command  # noqa: E402

reset_commands()

# Устанавливаем только инструмент uv; зависимости проекта устанавливает uv.lock.

UV_VERSION = "0.12.15"
if shutil.which("uv") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", f"uv=={UV_VERSION}"], check=True
    )
UV = shutil.which("uv")
ENV = {
    "UV_PROJECT_ENVIRONMENT": "/tmp/sleep-edf-usleep-rbd-venv",
    "UV_CACHE_DIR": "/tmp/sleep-pretrained-uv-cache",
    "UV_PYTHON_INSTALL_DIR": "/tmp/sleep-pretrained-python",
    "PYTHONPATH": str(PROJECT_ROOT),
    "MPLBACKEND": "Agg",  # Backend только для дочернего CLI, не Kaggle-ядра.
}
UV_PROJECT = PROJECT_ROOT / "environments/pretrained"
print("Ядро notebook:", sys.version)
subprocess.run(["nvidia-smi"], check=True)
run_command(
    [
        UV,
        "sync",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "--python",
        "3.12.12",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)
BASE = [
    UV,
    "run",
    "--project",
    str(UV_PROJECT),
    "--locked",
    "--no-dev",
    "python",
    "-m",
    "src.usleep_rbd_run",
]
ENV.update(
    {
        k: str(THREADS)
        for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
    }
)
run_command(
    [
        UV,
        "run",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "python",
        "-c",
        "from src.pretrained import hardware; import json; print(json.dumps(hardware("
        + repr(DEVICE)
        + "),indent=2))",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)

In [ ]:
import hashlib
from zipfile import ZipFile

import pandas as pd
from IPython.display import display


def unpack(path):
    path = Path(path)
    if path.is_dir():
        return path
    key = hashlib.sha256(
        (str(path) + str(path.stat().st_size) + str(path.stat().st_mtime_ns)).encode()
    ).hexdigest()[:16]
    target = Path("/tmp/sleep-edf-usleep-rbd-inputs") / key
    if (target / ".unpacked").exists():
        return target
    target.mkdir(parents=True, exist_ok=True)
    with ZipFile(path) as z:
        for info in z.infolist():
            dest = (target / info.filename).resolve()
            if (
                not dest.is_relative_to(target.resolve())
                or (info.external_attr >> 16) & 0o170000 == 0o120000
            ):
                raise ValueError("Небезопасный путь в ZIP")
            z.extract(info, target)
    (target / ".unpacked").touch()
    return target


roots = [
    p.parent
    for p in Path("/kaggle/input").rglob("MANIFEST.json")
    if (p.parent / "labels.csv").is_file() and (p.parent / "epochs/val").is_dir()
]
if len(roots) != 1:
    raise RuntimeError(f"Нужен один исходный full_psg_v1; найдено: {roots}")
DATA_ROOT = roots[0]
if INPUT_ROOT is None:
    found = [
        p.parent
        for p in Path("/kaggle/input").rglob("bundle.json")
        if json.loads(p.read_text()).get("kind") == "usleep_rbd_inputs_v1"
    ]
    if not found:
        for archive in Path("/kaggle/input").rglob("*.zip"):
            with ZipFile(archive) as z:
                if (
                    "bundle.json" in z.namelist()
                    and json.loads(z.read("bundle.json")).get("kind")
                    == "usleep_rbd_inputs_v1"
                ):
                    found.append(unpack(archive))
    if len(found) != 1:
        raise RuntimeError(
            "Подключите один новый sleep-edf-usleep-rbd-inputs.zip или укажите INPUT_ROOT явно"
        )
    INPUT_ROOT = found[0]
INPUT_ROOT = unpack(INPUT_ROOT)
if RESUME_ROOT is not None:
    previous = unpack(RESUME_ROOT)
    for path in previous.rglob("*"):
        if (
            not path.is_file()
            or path.name == ".unpacked"
            or (path.name.endswith(".lock") and path.name != "uv.lock")
        ):
            continue
        target = OUTPUT_ROOT / path.relative_to(previous)
        if target.exists():
            if (
                hashlib.sha256(target.read_bytes()).digest()
                != hashlib.sha256(path.read_bytes()).digest()
            ):
                raise ValueError(
                    f"Resume conflict; сохранён существующий файл: {target}"
                )
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)
COMMON = [
    "--data-root",
    str(DATA_ROOT),
    "--input-root",
    str(INPUT_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--device",
    DEVICE,
    "--threads",
    str(THREADS),
    "--group",
    RUN_GROUP,
]
print("DATA_ROOT:", DATA_ROOT, "\nINPUT_ROOT:", INPUT_ROOT)
print("Официальные веса и сохранённые113 уже в bundle. Старые модели не запускаются.")

In [ ]:
from src.notebook import reset_commands

reset_commands()
command = [*BASE, *COMMON]
if SMOKE_ONLY:
    command += ["--smoke", "--smoke-epochs", str(SMOKE_EPOCHS)]
try:
    run_command(command, cwd=PROJECT_ROOT, env=ENV)
except subprocess.CalledProcessError:
    print(
        "Есть ошибка; скачайте накопительный ZIP следующей ячейкой. Готовые записи сохранены."
    )
namespace = OUTPUT_ROOT / "smoke" if SMOKE_ONLY else OUTPUT_ROOT
report_path = namespace / "usleep_rbd_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    display(pd.DataFrame(report["results"]))
    print("Лучший только по val:", report["best_validation"])
    print(
        "Все запрошенные готовы:",
        report["all_requested_complete"],
        "Ошибки:",
        report["failures"],
    )
check = namespace / "standalone_validation_check.json"
if check.exists():
    print("Проверка чистой U-Sleep:", json.loads(check.read_text()))

ZIP накопительный: результаты, probabilities, метрики, журнал и snapshot кода. Input-веса отдельно уже сохранены в подключённом Dataset.
После остановки можно выгрузить ZIP и возобновить ту же версию с RESUME_ROOT.


In [ ]:
from datetime import UTC, datetime

from IPython.display import FileLink, display

from src.notebook import reset_commands

reset_commands()
stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
archive = Path("/kaggle/working") / f"sleep-edf-usleep-rbd-results-{stamp}.zip"
run_command(
    [*BASE, "--output-root", str(OUTPUT_ROOT), "--export", str(archive)],
    cwd=PROJECT_ROOT,
    env=ENV,
)
display(
    FileLink(
        str(archive.relative_to(Path.cwd())),
        result_html_prefix="Скачать все результаты и сабмишны одним ZIP: ",
    )
)